## Inicialização 

In [1]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
CACHE_DIR = PROJECT_ROOT / ".cache"
MPL_CACHE_DIR = CACHE_DIR / "matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(MPL_CACHE_DIR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd


I0000 00:00:1776027458.990702    3489 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776027459.183552    3489 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776027461.380647    3489 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Carregue os dados

O conjunto de dados é armazenado na pasta `/datasets/faces/`, onde você pode encontrar "
- A pasta `final_files` com 7,6k fotos "
- O arquivo `labels.csv` com rótulos, com duas colunas: `file_name` e `real_age` 

Dado que o número de arquivos de imagem é bastante alto, é aconselhável evitar a leitura de todos de uma vez, o que consumiria muito recursos computacionais. Recomendamos que você crie um gerador com o ImageDataGenerator. Este método foi explicado no Capítulo 3, Lição 7 deste curso.

O arquivo de rótulo pode ser carregado como um arquivo CSV normal.

In [2]:


# crie dois geradores em vez de um: train_datagen e valid_datagen
'''
datagen = ImageDataGenerator(validation_split=0.25, rescale=1/255.)

train_datagen = ImageDataGenerator(
    validation_split=0.25, rescale=1.0 / 255, horizontal_flip=True, vertical_flip = True, width_shift_range=0.2, height_shift_range=0.2, rotation_range=90, 
)

validation_datagen = ImageDataGenerator(
    validation_split=0.25, rescale=1.0 / 255
)


train_datagen_flow = datagen.flow_from_directory(
    '/datasets/fruits_small/',
    target_size=(150, 150),
    batch_size=16,
    class_mode='sparse',
    subset='training',
    seed=12345)

val_datagen_flow = datagen.flow_from_directory(
    '/datasets/fruits_small/',
    target_size=(150, 150),
    batch_size=16,
    class_mode='sparse',
    subset='validation',
    seed=12345)

features, target = next(train_datagen_flow)

# mostra 16 imagens
fig = plt.figure(figsize=(10,10))
for i in range(16):
    fig.add_subplot(4, 4, i+1)
    plt.imshow(features[i])
	# remova os eixos e coloque as imagens mais próximas umas das outras para uma saída mais compacta
    plt.xticks([])
    plt.yticks([])
    plt.tight_layout()

'''
def check_image_sizes(directory):
    sizes = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, file)
                img = Image.open(img_path)
                sizes.append(img.size)
    return sizes

# Verificar tamanhos no seu dataset
sizes = check_image_sizes('../data/faces/final_files/')
print(f"Primeiros 5 tamanhos: {sizes[:5]}")
print(f"Tamanhos únicos: {set(sizes)}")



# Converter sizes para DataFrame
df_sizes = pd.DataFrame(sizes, columns=['width', 'height'])

# Obter estatísticas descritivas
print("Estatísticas descritivas dos tamanhos das imagens:")
print(df_sizes.describe())

# Informações adicionais úteis
print(f"Total de imagens: {len(sizes)}")

print(df_sizes.drop_duplicates().sort_values(['width', 'height']))

Primeiros 5 tamanhos: [(287, 287), (956, 955), (345, 345), (381, 381), (453, 453)]
Tamanhos únicos: {(277, 277), (1108, 1108), (580, 580), (446, 500), (1159, 1159), (310, 310), (343, 344), (631, 631), (901, 900), (73, 74), (613, 613), (2293, 2293), (646, 646), (106, 107), (409, 410), (136, 135), (679, 679), (118, 117), (439, 438), (377, 377), (982, 982), (712, 712), (1000, 999), (151, 150), (184, 184), (410, 410), (443, 444), (475, 476), (454, 453), (217, 217), (505, 504), (206, 207), (999, 746), (250, 250), (538, 537), (509, 510), (797, 797), (815, 845), (520, 519), (207, 208), (779, 779), (527, 527), (239, 240), (251, 251), (1070, 1071), (830, 830), (2189, 2189), (240, 241), (812, 812), (284, 284), (863, 863), (302, 301), (317, 317), (47, 47), (335, 334), (911, 912), (350, 350), (1181, 1181), (929, 929), (80, 80), (944, 945), (383, 383), (1214, 1214), (1221, 1222), (146, 147), (125, 124), (416, 416), (675, 676), (176, 175), (114, 114), (1007, 1006), (158, 157), (726, 727), (449, 449)

## EDA

In [3]:
pequenas = df_sizes[(df_sizes['width'] < 100) | (df_sizes['height'] < 100)]
print(len(pequenas))

196


### Conclusões

## Modelagem 

Defina as funções necessárias para treinar seu modelo na plataforma GPU e construa um único script contendo todas elas junto com a seção de inicialização.

Para facilitar essa tarefa, você pode defini-las neste notebook e executar um código pronto na próxima seção para compor o script automaticamente.
As definições abaixo também serão verificadas pelos revisores do projeto, para que possam entender como você construiu o modelo.

In [4]:
import tensorflow as tf

import pandas as pd

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam

In [5]:
# Aplicar o mesmo filtro de imagens pequenas
def filter_small_images(df):
    valid_files = []
    for _, row in df.iterrows():
        img_path = f"../data/faces/final_files/{row['file_name']}"
        try:
            img = Image.open(img_path)
            width, height = img.size
            if width >= 100 and height >= 100:
                valid_files.append(row['file_name'])
        except:
            continue
    return df[df['file_name'].isin(valid_files)]



In [6]:

def load_train(path):
    """
    Carrega a parte de treinamento do conjunto de dados a partir do caminho
    """
    # Carregar labels
    labels_df = pd.read_csv('../data/faces/labels.csv')

    
    # Aplicar filtro
    filtered_labels = filter_small_images(labels_df)
    
    # Criar gerador com data augmentation
    train_datagen = ImageDataGenerator(
        validation_split=0.25,
        rescale=1.0/255,
        horizontal_flip=True,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2
    )
    
    train_gen_flow = train_datagen.flow_from_dataframe(
        dataframe=filtered_labels,
        directory='/datasets/faces/final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),  # Mudança para 224x224
        batch_size=32,
        class_mode='raw',
        subset='training',
        seed=12345
    )
    
    return train_gen_flow

In [7]:

def load_test(path):
    """
    Carrega a parte de validação/teste do conjunto de dados a partir do caminho
    """
    # Carregar labels
    labels_df = pd.read_csv('/datasets/faces/labels.csv')
    # NÃO serão filtradas as imagens de baixa qualidade já que elas existem no contexto.
    
    # Criar gerador SEM data augmentation para validação
    test_datagen = ImageDataGenerator(
        validation_split=0.25,
        rescale=1.0/255
    )
    
    test_gen_flow = test_datagen.flow_from_dataframe(
        dataframe=labels_df,
        directory='/datasets/faces/final_files/',
        x_col='file_name',
        y_col='real_age',
        target_size=(224, 224),
        batch_size=32,
        class_mode='raw',
        subset='validation',  # Esta é a diferença principal
        seed=12345
    )
    
    return test_gen_flow

In [8]:
def create_model(input_shape):
    
    """
   Define o modelo
    """
    
    # coloque seu código aqui

  #  retorna o modelo


In [9]:
def train_model(model, train_data, test_data, batch_size=None, epochs=20,
                steps_per_epoch=None, validation_steps=None):

    """
    Treina o modelo de acordo com os parâmetros
    """
    
    # coloque seu código aqui

 #  retorna o modelo


## Preparar o Script para a Execução na plataforma GPU

Dado que você definiu as funções necessárias, você pode compor um script para a plataforma GPU, baixá-lo através do menu "Arquivo|Abrir..." e carregá-lo posteriormente para execução na plataforma GPU.

Nota: O script também deve incluir a seção de inicialização. Um exemplo disso é mostrado abaixo.

In [10]:
# preparar um script para ser executado na plataforma GPU


init_str = """
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
CACHE_DIR = PROJECT_ROOT / '.cache'
MPL_CACHE_DIR = CACHE_DIR / 'matplotlib'
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ['MPLCONFIGDIR'] = str(MPL_CACHE_DIR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
"""

import inspect

with open('run_model_on_gpu.py', 'w') as f:
    
    f.write(init_str)
    f.write('\n\n')
        
    for fn_name in [load_train, load_test, create_model, train_model]:
        
        src = inspect.getsource(fn_name)
        f.write(src)
        f.write('\n\n')

### Resultado

Coloque a saída da plataforma GPU como uma célula Markdown aqui.

## Conclusão

# Checklist

- [ ]  O notebook foi aberto 
- [ ]  O código está livre de erros 
- [ ]  As células com código foram organizadas por ordem de execução 
- [ ]  A análise exploratória dos dados foi realizada - [ ]  Os resultados da análise exploratória dos dados são apresentados no caderno final - [ ]  O valor EAM do modelo não é superior a 8 
- [ ]  O código de treinamento do modelo foi copiado para o notebook final 
- [ ]  A saída do treinamento do modelo foi copiada para o notebook final 
- [ ]  As conclusões foram fornecidas com base nos resultados do treinamento do modelo